# Workshop 03: The Forecaster\x27s Toolbox — Benchmarks, Residual Forensics & Prediction Intervals
**Machine Learning II: Forecasting Economic Time Series** — *Lazarski University*  
*Author:* Leonardo Di Caterina (`l.dicaterina@lazarski.pl`)  
*Theoretical Reference:* Hyndman & Athanasopoulos (2021), *Forecasting: Principles and Practice* (FPP3), Chapter 5.

---

### Workshop Objectives
1. **Implement 4 Vectorized Benchmark Forecasters**: Mean, Naïve, Seasonal Naïve, and Drift methods with strict temporal index alignment.
2. **Compute One-Step In-Sample Residuals & Table 5.2 Multi-Step Standard Errors**: Understand parameter degrees of freedom ($K$) and horizon scaling $\sigma\sqrt{h}$.
3. **Quantify Prediction Uncertainty**: Build analytical Gaussian intervals and non-parametric bootstrapped residual sample path ensembles.
4. **Automate Residual Diagnostic Auditing (`check_residuals`)**: Verify unbiasedness, Bartlett ACF bounds ($\pm 1.96/\sqrt{T}$), and the Ljung-Box portmanteau test.
5. **Backtest Empirical Coverage**: Audit whether nominal 95% intervals achieve true 95% empirical calibration across rolling windows.
6. **Implement Decomposition Forecasting (STLF)**: Decompose non-stationary seasonal series with STL, extrapolate components, and reseasonalise.


In [ ]:
# =============================================================================
# Global Imports & Reproducibility Setup
# =============================================================================
import os
import sys
from pathlib import Path
from typing import Dict, List, Any, Union, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL

# Enforce deterministic reproducibility (RULES.md §3)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Matplotlib styling for high-legibility econometric diagnostics
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
print("Workshop 03 environment initialized. Data directory:", DATA_DIR.resolve())


---
## Part 1: Foundational Benchmark Forecasting Engines (35 Minutes)

### Theoretical Foundation (FPP3 §5.2)
Before building complex econometric or deep learning models, we must establish baseline hurdles. If a complex algorithm cannot outperform these simple heuristics, it destroys economic value:

1. **Mean Method**: $\hat{y}_{T+h|T} = \bar{y} = \frac{1}{T}\sum_{t=1}^T y_t$. Optimal under squared error loss for stationary white noise around a fixed mean.
2. **Naïve Method**: $\hat{y}_{T+h|T} = y_T$. Sets all future forecasts to the most recently observed value; optimal for a pure random walk ($y_t = y_{t-1} + \varepsilon_t$).
3. **Seasonal Naïve Method**: $\hat{y}_{T+h|T} = y_{T+h - m(k+1)}$, where $m$ is seasonal period and $k = \lfloor(h-1)/m\rfloor$. Repeats the last observed seasonal cycle.
4. **Drift Method**: $\hat{y}_{T+h|T} = y_T + h\left(\frac{y_T - y_1}{T - 1}\right)$. Equivalent to drawing a line between the first and last historical observations and extrapolating it.


In [ ]:
def load_and_index_series(
    filepath: str,
    date_col: str,
    val_col: str,
    freq: Optional[str] = None
) -> pd.Series:
    """
    Ingests a raw time series CSV, enforces an immutable DatetimeIndex and calendar frequency.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def _generate_future_index(y_train: pd.Series, h: int) -> pd.DatetimeIndex:
    """Generates a DatetimeIndex of length h extending beyond the historical sample."""
    freq = y_train.index.freq
    if freq is None:
        inferred = pd.infer_freq(y_train.index)
        freq = inferred if inferred is not None else "D"
    
    last_date = y_train.index[-1]
    full_range = pd.date_range(start=last_date, periods=h + 1, freq=freq)
    return full_range[1:]


def forecast_mean(y_train: pd.Series, h: int) -> pd.Series:
    """Extrapolates sample mean horizontally across future horizons h >= 1."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def forecast_naive(y_train: pd.Series, h: int) -> pd.Series:
    """Sets all future forecasts to the final observed value y_T."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def forecast_snaive(y_train: pd.Series, h: int, m: int) -> pd.Series:
    """Sets each forecast to the last value from the identical season."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def forecast_drift(y_train: pd.Series, h: int) -> pd.Series:
    """Linearly extrapolates the historical boundary slope from y_1 to y_T."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Benchmark Showdown on Australian Quarterly Beer (FPP3 Figure 5.7)
beer = load_and_index_series(str(DATA_DIR / "aus_beer.csv"), "observation_date", "beer_production", freq="QS")
train_beer = beer.loc[:"2006-10-01"]
test_beer = beer.loc["2007-01-01":]
h_beer = len(test_beer)

fc_mean = forecast_mean(train_beer, h=h_beer)
fc_naive = forecast_naive(train_beer, h=h_beer)
fc_snaive = forecast_snaive(train_beer, h=h_beer, m=4)
fc_drift = forecast_drift(train_beer, h=h_beer)

plt.figure(figsize=(12, 5))
plt.plot(train_beer.index[-32:], train_beer.iloc[-32:], label="Historical Training", color="#1A3A6B", lw=2)
plt.plot(test_beer.index, test_beer.values, label="Actual Test Data", color="black", lw=2.5, ls="--")
plt.plot(fc_mean.index, fc_mean.values, label="Mean Forecast", color="#2E6B9E", lw=1.8)
plt.plot(fc_naive.index, fc_naive.values, label="Naïve Forecast", color="#C99700", lw=1.8)
plt.plot(fc_snaive.index, fc_snaive.values, label="Seasonal Naïve (m=4)", color="#107C41", lw=2.2)
plt.plot(fc_drift.index, fc_drift.values, label="Drift Forecast", color="#D83B01", lw=1.8)

plt.title("Australian Quarterly Beer Production: Benchmark Showdown (Replicating FPP3 Fig 5.7)")
plt.xlabel("Quarter")
plt.ylabel("Megalitres")
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()


---
## Part 2: Fitted Values, Residuals & Multi-Step Standard Errors (20 Minutes)

### Theoretical Foundation (FPP3 §5.3–§5.5)
- **One-Step In-Sample Residuals**: $e_t = y_t - \hat{y}_{t|t-1}$.
  - For Mean: $\hat{y}_t = \bar{y}$ (0 NaNs).
  - For Naïve: $\hat{y}_t = y_{t-1}$ (first residual is undefined, $M=1$ NaN).
  - For Seasonal Naïve: $\hat{y}_t = y_{t-m}$ (first $m$ residuals undefined, $M=m$ NaNs).
  - For Drift: $\hat{y}_t = y_{t-1} + \hat{c}$ ($M=1$ NaN).
- **Degrees of Freedom**: $\hat{\sigma} = \sqrt{\frac{1}{T - K - M}\sum_{t=1}^T e_t^2}$, where $K$ is the number of estimated parameters ($K=0$ for Naïve/SNaïve, $K=1$ for Mean/Drift).
- **Table 5.2 Multi-Step Scaling**:
  $$\sigma_h = \begin{cases} \hat{\sigma}\sqrt{1 + 1/T} & \text{Mean} \\ \hat{\sigma}\sqrt{h} & \text{Naïve} \\ \hat{\sigma}\sqrt{k+1} & \text{Seasonal Naïve} \\ \hat{\sigma}\sqrt{h(1 + h/(T-1))} & \text{Drift} \end{cases}$$


In [ ]:
def compute_benchmark_residuals(
    y_train: pd.Series,
    method: str,
    m: Optional[int] = None
) -> pd.Series:
    """Computes one-step in-sample residuals e_t = y_t - y_hat_{t|t-1}."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def compute_forecast_sd(
    residuals: pd.Series,
    method: str,
    h: int,
    T: int,
    m: Optional[int] = None
) -> float:
    """Computes multi-step standard error sigma_h using exact FPP3 Table 5.2 formulas."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


def compute_prediction_intervals(
    y_hat: pd.Series,
    residuals: pd.Series,
    method: str,
    levels: List[int] = [80, 95],
    m: Optional[int] = None
) -> pd.DataFrame:
    """Constructs analytical Gaussian prediction intervals across horizons h=1..len(y_hat)."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


---
## Part 3: Distribution-Free Bootstrapped Prediction Intervals (20 Minutes)

### Theoretical Foundation (FPP3 §5.5)
When normality fails (e.g. fat-tailed financial shocks or skewed demand), Gaussian analytical intervals severely under-cover true outcomes. Instead of assuming $e_t \sim N(0, \sigma^2)$, we sample historical residuals with replacement to simulate $B = 1,000$ sequential future sample paths:
$$y^*_{T+1} = y_T + e^*_{T+1}, \quad y^*_{T+2} = y^*_{T+1} + e^*_{T+2}, \quad \dots$$
Empirical percentiles (e.g. 2.5% and 97.5%) form the prediction interval bounds without any parametric assumptions.


In [ ]:
def bootstrap_prediction_intervals(
    y_train: pd.Series,
    method: str,
    h: int,
    B: int = 1000,
    levels: List[int] = [80, 95],
    m: Optional[int] = None,
    random_state: int = 42
) -> pd.DataFrame:
    """Simulates B future sample paths and computes empirical quantile prediction intervals."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


---
## Part 4: Automated Residual Diagnostics Suite (`check_residuals`) (25 Minutes)

### Theoretical Foundation (FPP3 §5.4)
Residual diagnostics audit whether a model has left predictable signal unexploited:
- **Unbiasedness ($E[e_t] = 0$)**: Evaluated via a single-sample t-test ($H_0: \mu = 0$).
- **Portmanteau Ljung-Box Test**: Evaluates whether the first $\ell$ autocorrelations differ jointly from zero:
  $$Q^* = T(T+2)\sum_{k=1}^\ell (T-k)^{-1} r_k^2 \quad \sim \quad \chi^2_{\ell - K}$$
  Degrees of freedom must be adjusted for estimated parameters ($DoF = \ell - K$).
- **Normality**: Evaluated via skewness, kurtosis, and the Jarque-Bera test.


In [ ]:
def check_residuals(
    residuals: pd.Series,
    lags: Optional[int] = None,
    seasonal_m: Optional[int] = None,
    K: int = 0
) -> Dict[str, Any]:
    """
    Executes an automated four-stage residual forensic diagnostic battery.
    Replicates FPP3 Section 5.4 and gg_tsresiduals.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Forensic Demonstration on Google Closing Stock (Replicating FPP3 Fig 5.13)
google = load_and_index_series(str(DATA_DIR / "gafa_stock_google.csv"), "observation_date", "adj_close", freq="B")
res_google = compute_benchmark_residuals(google, method="naive")
diag_google = check_residuals(res_google, lags=10, K=0)

print("--- Google Stock Residual Diagnostics ---")
for k, v in diag_google.items():
    print(f"{k:18s}: {v}")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# 1. Top Panel: Time Plot
ax_top = plt.subplot2grid((2, 2), (0, 0), colspan=2)
ax_top.plot(res_google.index, res_google.values, color="#1A3A6B", lw=1.2)
ax_top.axhline(0, color="black", ls="--", alpha=0.6)
ax_top.set_title("Google Stock Residuals: Time Plot (Inspecting Variance & Outliers)")
ax_top.set_ylabel("Residual ($)")

# 2. Bottom Left: ACF Plot with Bartlett Bounds
ax_acf = plt.subplot2grid((2, 2), (1, 0))
T_g = len(res_google.dropna())
bartlett = 1.96 / np.sqrt(T_g)
lags_range = range(1, 21)
acf_vals = [res_google.dropna().autocorr(lag=l) for l in lags_range]
ax_acf.vlines(lags_range, [0], acf_vals, color="#1A3A6B", lw=2)
ax_acf.scatter(lags_range, acf_vals, color="#1A3A6B", zorder=3)
ax_acf.axhline(bartlett, color="#2E6B9E", ls="--", label=f"95% Bartlett (±{bartlett:.3f})")
ax_acf.axhline(-bartlett, color="#2E6B9E", ls="--")
ax_acf.axhline(0, color="black", lw=0.8)
ax_acf.set_title("Sample Autocorrelation Function (ACF)")
ax_acf.set_xlabel("Lag")
ax_acf.set_ylabel("ACF")
ax_acf.legend(frameon=True)

# 3. Bottom Right: Histogram & Normal Fit
ax_hist = plt.subplot2grid((2, 2), (1, 1))
sns.histplot(res_google.dropna(), kde=True, ax=ax_hist, color="#1A3A6B", stat="density")
ax_hist.set_title("Residual Distribution vs Gaussian Curve")
ax_hist.set_xlabel("Residual ($)")

plt.tight_layout()
plt.show()


---
## Part 5: Expanding-Window Backtest & Empirical Coverage Auditing (20 Minutes)

### Theoretical Foundation (FPP3 §5.5)
A prediction interval claiming 95% nominal coverage is useless if empirical test coverage is only 80%. We audit calibration across rolling temporal origins:
$$\text{Coverage}_h = \frac{1}{N}\sum_{t=1}^N \mathbb{I}\left(L_{t+h|t} \le y_{t+h} \le U_{t+h|t}\right)$$


In [ ]:
def evaluate_empirical_coverage(
    y_series: pd.Series,
    method: str,
    initial_train_size: int,
    h: int,
    level: int = 95,
    m: Optional[int] = None
) -> float:
    """Conducts an expanding-origin backtest to audit empirical coverage at horizon h."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


---
## Part 6: Decomposition Forecasting (STLF Pipeline) (15 Minutes)

### Theoretical Foundation (FPP3 §5.7)
Decomposition turns a difficult non-stationary seasonal series into two simple forecasting problems:
1. Decompose $y_t = S_t + A_t$ via STL (where $A_t = T_t + R_t$ is seasonally adjusted).
2. Project seasonal component $\hat{S}_{T+h}$ using `forecast_snaive`.
3. Project seasonally adjusted component $\hat{A}_{T+h}$ using `forecast_drift` or `forecast_naive`.
4. Reseasonalise by summing the components: $\hat{y}_{T+h} = \hat{S}_{T+h} + \hat{A}_{T+h}$.


In [ ]:
def forecast_stlf(
    y_train: pd.Series,
    h: int,
    m: int,
    method_trend: str = "drift"
) -> pd.Series:
    """Forecasting with decomposition (STLF)."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: STLF on US Monthly Retail Sales (RSAFS)
rsafs = load_and_index_series(str(DATA_DIR / "RSAFS.csv"), "observation_date", "RSAFSNA", freq="MS")
train_r = rsafs.loc["2010-01-01":"2022-12-01"]
test_r = rsafs.loc["2023-01-01":"2024-06-01"]
h_r = len(test_r)

fc_stlf = forecast_stlf(train_r, h=h_r, m=12, method_trend="drift")

plt.figure(figsize=(12, 5))
plt.plot(train_r.index[-36:], train_r.values[-36:], label="Historical RSAFS", color="#1A3A6B", lw=2)
plt.plot(test_r.index, test_r.values, label="Actual Test Data", color="black", lw=2.5, ls="--")
plt.plot(fc_stlf.index, fc_stlf.values, label="STLF Forecast (STL + SNaive + Drift)", color="#107C41", lw=2.2)

plt.title("Forecasting US Retail Sales with Decomposition (STLF Pipeline)")
plt.xlabel("Month")
plt.ylabel("Million USD")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


---
## Part 7: Practical Syllabus Interview Clinic

#### Question 1: Implement a seasonal naive forecast in a few lines of Python.
**Model Answer:**  
Extract the final $m$ observations of the training series (`y_train.iloc[-m:].values`) and project them forward using modulo arithmetic: `[last_season[(i - 1) % m] for i in range(1, h + 1)]`. Wrap in a `pd.Series` with a properly constructed future `pd.DatetimeIndex`. This requires zero parameter estimation and forms the non-negotiable benchmark for all seasonal economic series.

#### Question 2: Which residual plots and tests do you check, and what does each failure look like?
**Model Answer:**  
We check the three-panel diagnostic battery (time plot, ACF, histogram) and the Ljung-Box portmanteau test:
1. **Time Plot**: Inspect for non-constant variance (heteroskedastic funnels) or abrupt shocks. A failure looks like an expanding trumpet shape.
2. **ACF Plot**: Inspect for autocorrelations outside Bartlett bounds ($\pm 1.96/\sqrt{T}$). A failure looks like significant spikes at lag 1 (momentum) or seasonal lags (unmodeled seasonality).
3. **Histogram**: Inspect for symmetry and bell-curve decay. A failure looks like severe skewness or heavy fat tails.
4. **Ljung-Box Test**: Evaluates joint autocorrelation up to lag $\ell$. If $p < 0.05$, residuals contain unexploited information and fail the white noise requirement.

#### Question 3: How do you obtain prediction intervals from a benchmark and check their coverage?
**Model Answer:**  
- **Analytical Intervals**: Compute residual variance $\hat{\sigma}^2 = \frac{1}{T - K - M}\sum e_t^2$. Scale $\hat{\sigma}$ across horizons using Table 5.2 formulas (e.g. $\hat{\sigma}\sqrt{h}$ for Naïve, $\hat{\sigma}\sqrt{k+1}$ for Seasonal Naïve). Construct bounds as $\hat{y} \pm z \sigma_h$.
- **Bootstrapped Intervals**: When normality fails, sample residuals with replacement sequentially over $B=1,000$ paths and compute empirical percentiles (2.5% and 97.5%).
- **Coverage Audit**: Conduct an expanding-origin backtest. Count the fraction of test actuals falling inside $[L, U]$. If nominal 95% intervals capture $\ge 95\%$ of outcomes, calibration is verified.
